# Research Synthesis Agent

An agentic system that searches ArXiv and Semantic Scholar for academic papers,
extracts key findings, checks for contradictions across studies, and synthesizes
everything into a research briefing — orchestrated via OpenAI function calling.

## Why an Agent?

Research synthesis is **inherently exploratory**: you start with a question,
discover relevant papers, notice that two studies contradict each other, and
need to dig deeper into each one. A static pipeline would fetch papers and
summarize them in a fixed order. An agent can **adapt**: if the initial ArXiv
search returns too few results, it can broaden the query; if it detects a
contradiction, it can fetch more details about the conflicting papers; and
it can use Semantic Scholar to get citation counts and influence metrics
before deciding which findings to trust.

In [ ]:
%pip install openai requests pandas feedparser --quiet

In [ ]:
import json, textwrap, time, re
import requests
import feedparser
import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## Tool 1 — `search_arxiv`

Searches the ArXiv API for papers matching a query, returning titles,
authors, abstracts, and ArXiv IDs.

In [ ]:
def search_arxiv(query: str, max_results: int = 5) -> str:
    """Search ArXiv for papers matching a query. Returns up to max_results papers."""
    url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": min(max_results, 10),
        "sortBy": "relevance",
        "sortOrder": "descending",
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        feed = feedparser.parse(resp.text)

        results = []
        for entry in feed.entries:
            # Extract ArXiv ID from the URL
            arxiv_id = entry.id.split("/abs/")[-1]
            authors = [a.get("name", "") for a in entry.get("authors", [])]
            results.append({
                "arxiv_id": arxiv_id,
                "title": entry.title.replace("\n", " ").strip(),
                "authors": authors[:5],  # first 5 authors
                "published": entry.get("published", "")[:10],
                "abstract": entry.summary.replace("\n", " ").strip()[:500],
                "categories": [t["term"] for t in entry.get("tags", [])],
                "pdf_url": f"https://arxiv.org/pdf/{arxiv_id}",
            })

        return json.dumps(results, indent=2)
    except Exception as exc:
        return json.dumps({"error": str(exc)})

## Tool 2 — `get_paper_details`

Uses the Semantic Scholar API to get citation count, influential citations,
and other metadata for a paper identified by its ArXiv ID.

In [ ]:
def get_paper_details(arxiv_id: str) -> str:
    """Get citation count, venue, and influence metrics from Semantic Scholar
    for a paper identified by its ArXiv ID."""
    url = f"https://api.semanticscholar.org/graph/v1/paper/ArXiv:{arxiv_id}"
    params = {
        "fields": "title,year,citationCount,influentialCitationCount,venue,openAccessPdf,tldr"
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        if resp.status_code == 404:
            return json.dumps({"error": f"Paper ArXiv:{arxiv_id} not found on Semantic Scholar"})
        resp.raise_for_status()
        data = resp.json()
        result = {
            "arxiv_id": arxiv_id,
            "title": data.get("title"),
            "year": data.get("year"),
            "venue": data.get("venue"),
            "citation_count": data.get("citationCount"),
            "influential_citation_count": data.get("influentialCitationCount"),
            "tldr": data.get("tldr", {}).get("text") if data.get("tldr") else None,
        }
        return json.dumps(result, indent=2)
    except Exception as exc:
        return json.dumps({"error": str(exc)})

## Tool 3 — `extract_key_findings`

Uses the LLM to extract structured key findings from an abstract.

In [ ]:
def extract_key_findings(abstract: str) -> str:
    """Extract structured key findings from a paper abstract.
    Returns a JSON object with findings, methodology, and limitations."""
    prompt = textwrap.dedent(f"""\
        Analyze this research abstract and extract structured information.

        Abstract: {abstract}

        Return JSON with:
        - main_finding (str): The primary result or claim in one sentence.
        - methodology (str): How the study was conducted (one sentence).
        - key_metrics (list of str): Any quantitative results mentioned.
        - limitations (str or null): Any limitations mentioned or implied.
        - domain (str): The research domain/field.
        - confidence_level (str): "high", "medium", or "low" based on how
          specific and well-supported the claims appear.
    """)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

## Tool 4 — `check_contradictions`

Takes a list of findings from multiple papers and uses the LLM to identify
any contradictions, agreements, or gaps.

In [ ]:
def check_contradictions(findings_list: list[dict]) -> str:
    """Analyze a list of findings from multiple papers for contradictions,
    agreements, and gaps. Each finding dict should have at minimum:
    paper_title, main_finding."""
    findings_text = json.dumps(findings_list, indent=2)
    prompt = textwrap.dedent(f"""\
        You are a research analyst. Analyze these findings from multiple papers
        and identify relationships between them.

        Findings:
        {findings_text}

        Return JSON with:
        - contradictions (list of {{paper_a, paper_b, description}}): pairs of
          findings that contradict or conflict with each other.
        - agreements (list of {{papers (list of str), description}}): findings
          that support or reinforce each other.
        - gaps (list of str): important questions that none of the papers address.
        - overall_consensus (str): one-paragraph summary of the overall state
          of knowledge on this topic based on these papers.
        - confidence_assessment (str): how confident we can be in the consensus,
          given the evidence.
    """)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

## Tool Schema for OpenAI Function Calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_arxiv",
            "description": "Search ArXiv for papers matching a query. Returns titles, abstracts, authors, and ArXiv IDs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query for ArXiv"},
                    "max_results": {"type": "integer", "description": "Maximum papers to return (default 5, max 10)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_paper_details",
            "description": "Get citation count, venue, and influence metrics from Semantic Scholar for a paper by its ArXiv ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "arxiv_id": {"type": "string", "description": "ArXiv paper ID, e.g. 2301.00234"},
                },
                "required": ["arxiv_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_key_findings",
            "description": "Extract structured key findings, methodology, and limitations from a paper abstract.",
            "parameters": {
                "type": "object",
                "properties": {
                    "abstract": {"type": "string", "description": "The paper abstract text"},
                },
                "required": ["abstract"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_contradictions",
            "description": "Analyze findings from multiple papers for contradictions, agreements, and gaps.",
            "parameters": {
                "type": "object",
                "properties": {
                    "findings_list": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "paper_title": {"type": "string"},
                                "main_finding": {"type": "string"},
                                "methodology": {"type": "string"},
                                "key_metrics": {"type": "array", "items": {"type": "string"}},
                                "confidence_level": {"type": "string"},
                            },
                        },
                        "description": "List of findings from different papers",
                    }
                },
                "required": ["findings_list"],
            },
        },
    }
]

## Dispatcher

In [ ]:
TOOL_MAP = {
    "search_arxiv": search_arxiv,
    "get_paper_details": get_paper_details,
    "extract_key_findings": extract_key_findings,
    "check_contradictions": check_contradictions,
}

## Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a Research Synthesis agent. Your job is to help users understand
    the state of research on a topic by:

    1. Searching ArXiv for relevant papers.
    2. Getting citation metrics from Semantic Scholar for the most relevant papers.
    3. Extracting key findings from each paper's abstract.
    4. Checking for contradictions or agreements across papers.
    5. Synthesizing everything into a clear research briefing.

    Guidelines:
    - Search for 5-7 papers initially.
    - Get Semantic Scholar details for the top 3-4 most relevant papers.
    - Extract findings from at least 3 papers before checking contradictions.
    - Always call check_contradictions before writing the final briefing.
    - In the final briefing, cite papers by title and include citation counts
      to indicate influence.
    - Be honest about limitations and gaps in the evidence.
""")


def run_agent(user_message: str, max_turns: int = 15) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"\n--- Agent turn {turn + 1} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print("\n--- Agent finished ---")
            return msg.content

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  Calling {fn_name}({json.dumps(fn_args)[:100]})")
            fn = TOOL_MAP.get(fn_name)
            result = fn(**fn_args) if fn else json.dumps({"error": f"Unknown tool: {fn_name}"})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            # Be polite to APIs
            time.sleep(0.5)

    return "Agent reached maximum turns without finishing."

## Example Execution — LLM Reasoning Capabilities

We ask the agent to synthesize recent research on reasoning abilities in
large language models.

In [ ]:
report = run_agent(
    "I want to understand the current state of research on reasoning capabilities "
    "in large language models. Specifically: Can LLMs actually reason, or are they "
    "just pattern matching? Search for recent papers, extract findings, check for "
    "contradictions in the literature, and give me a synthesis briefing."
)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Second Scenario — Retrieval-Augmented Generation

In [ ]:
report2 = run_agent(
    "Synthesize the latest research on Retrieval-Augmented Generation (RAG). "
    "I'm particularly interested in: how does chunking strategy affect retrieval "
    "quality? Are there papers that disagree on optimal chunk sizes? Search ArXiv, "
    "get citation metrics, extract findings, and check for contradictions. "
    "Give me a briefing I can share with my engineering team."
)

In [ ]:
Markdown(report2)

## Third Scenario — Narrow Technical Question

In [ ]:
report3 = run_agent(
    "Find papers about 'mixture of experts' architectures for language models. "
    "I want to know: do MoE models achieve better performance per FLOP than dense "
    "models? Get details on the top 3 most-cited papers and synthesize the findings."
)

In [ ]:
Markdown(report3)

## Results Analysis

| Aspect | Detail |
|--------|--------|
| **Data sources** | ArXiv API (live paper search), Semantic Scholar API (live citation metrics) |
| **Adaptive flow** | Agent searched, then selectively fetched details for high-relevance papers only |
| **Contradiction detection** | LLM-based analysis of extracted findings; useful for spotting disagreements |
| **Limitations** | ArXiv search relevance can be noisy; Semantic Scholar rate limits (100 req/5 min); findings extraction depends on abstract quality |

### Potential Extensions

1. **Full-text analysis** — download PDFs and extract findings from the full paper, not just the abstract.
2. **Citation graph** — add a tool to explore the citation network (who cites whom).
3. **Temporal analysis** — track how findings on a topic have evolved over time.
4. **Export** — add a tool that formats the briefing as a BibTeX-annotated document.

---
*Notebook by the LLM Course — Agentic Designs series*